In [ ]:
# ==============================================================================
# ÉTAPE 1 : Configuration de l'environnement
# ==============================================================================
# Installation de toutes les dépendances requises pour orchestrer le RAG
!pip install -q langchain torch transformers sentence-transformers datasets faiss-cpu
!pip install -q -U langchain-community datasets

# ==============================================================================
# ÉTAPE 2 : Chargement du Dataset
# ==============================================================================
import torch
from langchain_community.document_loaders import HuggingFaceDatasetLoader

# Définition de la source de connaissances (Databricks Dolly)
dataset_name = "databricks/databricks-dolly-15k"
page_content_column = "context"

# Initialisation du chargeur et récupération des données sous forme de documents LangChain
loader = HuggingFaceDatasetLoader(dataset_name, page_content_column)
data = loader.load()

print(f"Nombre total de documents chargés : {len(data)}")
print("Aperçu du premier document :")
print(data[0])

# ==============================================================================
# ÉTAPE 3 : Découpage des Documents (Chunking)
# ==============================================================================
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Découpage par blocs de 1000 caractères avec un chevauchement de 150 caractères
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)
docs = text_splitter.split_documents(data)

print(f"\nNombre total de chunks créés : {len(docs)}")
print("Exemple du premier chunk découpé :")
print(docs[0])

# ==============================================================================
# ÉTAPE 4 : Vectorisation (Embeddings)
# ==============================================================================
from langchain_community.embeddings import HuggingFaceEmbeddings

modelPath = "sentence-transformers/all-MiniLM-L6-v2"
model_kwargs = {'device': 'cpu'}
encode_kwargs = {'normalize_embeddings': False}

# Initialisation du modèle d'embeddings sémantiques
embeddings = HuggingFaceEmbeddings(
    model_name=modelPath,
    model_kwargs=model_kwargs,
    encode_kwargs=encode_kwargs
)

# Test rapide de bon fonctionnement
text = "This is a test document."
query_result = embeddings.embed_query(text)
print(f"\nDimension de l'embedding de test : {len(query_result)}")

# ==============================================================================
# ÉTAPE 5 : Création du Vector Store (FAISS)
# ==============================================================================
from langchain_community.vectorstores import FAISS

# Indexation des chunks de texte dans le moteur de recherche vectoriel FAISS
print("\nIndexation des documents dans FAISS... (Cette étape peut prendre un instant)")
db = FAISS.from_documents(docs, embeddings)
print("Indexation terminée avec succès !")

# ==============================================================================
# ÉTAPE 6 : Préparation du modèle de langage (LLM Pipeline)
# ==============================================================================
from transformers import AutoTokenizer, AutoModelForQuestionAnswering, pipeline
from langchain_community.llms import HuggingFacePipeline

model_name = "Intel/dynamic_tinybert"

# Chargement du tokenizer et du modèle configuré pour l'extraction de réponses (QA)
tokenizer = AutoTokenizer.from_pretrained(model_name, padding=True, truncation=True, max_length=512)
model = AutoModelForQuestionAnswering.from_pretrained(model_name)

# Création du pipeline natif Transformers
qa_pipeline = pipeline(
    "question-answering",
    model=model,
    tokenizer=tokenizer,
    return_tensors='pt'
)

# Enveloppement du pipeline Hugging Face pour le rendre compatible avec LangChain
llm = HuggingFacePipeline(
    pipeline=qa_pipeline,
    model_kwargs={"temperature": 0.7, "max_length": 512}
)

# ==============================================================================
# ÉTAPE 7 : Construction de la chaîne de recherche et génération (Retrieval QA)
# ==============================================================================
from langchain.chains import RetrievalQA

# Transformation du Vector Store en composant Retriever (recherche du Top 4 sémantique)
retriever = db.as_retriever(search_kwargs={"k": 4})

# Assemblage final du pipeline RAG avec la stratégie 'refine'
qa = RetrievalQA.from_chain_type(
    llm=llm, 
    chain_type="refine", 
    retriever=retriever, 
    return_source_documents=False
)

# ==============================================================================
# ÉTAPE 8 : Test et exécution du système RAG
# ==============================================================================
# Définition de la question cible
question = "What is cheesemaking?"

print(f"\nPose de la question au système RAG : '{question}'")
# Exécution de la chaîne et affichage de la réponse finale trouvée dans le contexte extrait
result = qa.invoke({"query": question})

print("\n--- RÉSULTAT GÉNÉRÉ ---")
print(result["result"])